# GPT-controlled research workflow, bridged by GitHub — Colab · Drive · Gemini · Google Cloud

**Google Cloud project:** `third-serenity-428705-g2`  
**GitHub (code bridge):** `Saehon/Saeid-Homayoun` → `main`  
**Canonical data archive:** Google Drive

## Architecture

1. **ChatGPT / GPT** uses the connected GitHub and Google Drive apps to plan research, edit reviewable scripts/notebooks, and inspect saved research outputs.
2. **GitHub** bridges ChatGPT's version-controlled code to Google Colab. GitHub does **not** transmit login sessions or automatically execute Colab code.
3. **Google Colab** runs notebook cells after your approval, mounting **Google Drive** and authenticating **Google Cloud** in its own session.
4. **Gemini in Colab** is the built-in Google coding assistant; optional Gemini **Vertex AI** inference is a separate Google Cloud API workflow.
5. **Colab writes test reports and approved research outputs to Google Drive**, which ChatGPT can review using its separate connected Drive app.
6. **`OPENAI_API_KEY` is OPTIONAL** and needed only when Python code inside Colab itself must invoke OpenAI models directly. Your connected ChatGPT-to-GitHub/Drive workflow does not use this secret.

**Run steps 1–6 below to verify Google authentication.** Do not assume the existence of a GitHub notebook means that a runtime is authorized. No billable model inference is performed here. No keys, sensitive datasets, or permissions are written to GitHub.


In [ ]:
import os, sys, json, subprocess, platform
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError

PROJECT_ID = "third-serenity-428705-g2"
DRIVE_ROOT = Path("/content/drive/MyDrive")
SETUP_DIR = DRIVE_ROOT / "Colab Notebooks" / "GOOGLE_AI_FIVE_SERVICE_INTEGRATION"
checks = {"Colab runtime": "PASS (code executed)",
          "Google Drive": "NOT TESTED", "Google Cloud": "NOT TESTED",
          "Gemini in Colab UI": "MANUAL TEST REQUIRED",
          "Gemini Vertex AI": "NOT TESTED", "OpenAI GPT SDK": "NOT TESTED",
          "OpenAI GPT live API": "NOT TESTED (approval required)"}
print("Project ID:", PROJECT_ID)
print("Python:", platform.python_version())


## 1 — Connect Google Drive

Approve the mount prompt. Writes a new **uniquely named** harmless test file in the setup folder and reads it back; never overwrites or deletes anything.


In [ ]:
from google.colab import drive
from uuid import uuid4
drive.mount("/content/drive")
if not DRIVE_ROOT.is_dir():
    raise RuntimeError("MyDrive did not mount.")
SETUP_DIR.mkdir(parents=True, exist_ok=True)
path = SETUP_DIR / ("drive_test_" + uuid4().hex[:12] + ".txt")
message = "Colab-to-Drive read/write verified"
with path.open("x", encoding="utf-8") as f:
    f.write(message)
assert path.read_text(encoding="utf-8") == message
checks["Google Drive"] = "PASS (write and readback)"
print("Drive PASS:", path)


## 2 — Authenticate Google Cloud

Requires Google sign-in. Verifies project selection and Application Default Credentials only, not cloud API permissions, Vertex AI access or billing status.


In [ ]:
from google.colab import auth
import google.auth
auth.authenticate_user(project_id=PROJECT_ID)
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID],
               check=True, capture_output=True, text=True)
actual_project = subprocess.run(["gcloud", "config", "get-value", "project"],
                                check=True, capture_output=True, text=True).stdout.strip()
credentials, adc_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"])
assert actual_project == PROJECT_ID and credentials is not None
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
checks["Google Cloud"] = "PASS (authenticated and project configured; APIs not tested)"
print("Google Cloud PASS:", actual_project)
print("ADC project metadata (may be empty/different):", adc_project)


## 3 — Install official Gemini and OpenAI Python libraries in Colab only

Installing libraries does **not** call an AI model.


In [ ]:
%pip -q install -U google-genai openai
print("Gemini and OpenAI client packages installed in Colab.")


## 4 — Prepare Gemini through Google Cloud Vertex AI

This is separate from the built-in **Gemini in Colab** assistant. Test the built-in assistant manually via its Gemini sidebar by asking a simple question. This cell creates an SDK client only; no inference is requested. Vertex API access may need Google IAM, API enablement and billing.


In [ ]:
from google import genai
try:
    gemini_client = genai.Client(
        vertexai=True, project=PROJECT_ID, location="global",
        credentials=credentials)
    checks["Gemini Vertex AI"] = "CLIENT READY (remote access NOT TESTED)"
    print("Gemini Vertex AI client initialized. No model calls made.")
except Exception as e:
    checks["Gemini Vertex AI"] = "BLOCKED (" + type(e).__name__ + ")"
    print("Gemini client setup blocked:", type(e).__name__)


## 5 — Prepare OpenAI GPT securely with Colab Secrets

From the Colab sidebar, open **Secrets** (key icon), add `OPENAI_API_KEY`, and grant notebook access. Create a key in your [OpenAI API account](https://platform.openai.com/api-keys); a ChatGPT subscription does not automatically provide API credits. This cell does not send an API request. Never paste your key into the notebook or GitHub.


In [ ]:
from google.colab import userdata
from openai import OpenAI
openai_client = None
try:
    openai_key = userdata.get("OPENAI_API_KEY")
except Exception:
    openai_key = None
if openai_key:
    openai_client = OpenAI(api_key=openai_key)
    checks["OpenAI GPT SDK"] = "CLIENT READY (remote API NOT TESTED)"
    print("OpenAI GPT client initialized from private Colab Secret.")
else:
    checks["OpenAI GPT SDK"] = "BLOCKED (add OPENAI_API_KEY in Colab Secrets)"
    print("To enable GPT: add OPENAI_API_KEY under Colab Secrets and grant access.")
del openai_key


## 6 — Save non-sensitive verification report to Drive

Only project ID, test status, Python and package versions are stored. No passwords, credentials, API keys or tokens are stored.


In [ ]:
def package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"
report = {
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "google_cloud_project": PROJECT_ID,
    "github_repository": "Saehon/Saeid-Homayoun",
    "python": platform.python_version(),
    "google-genai": package_version("google-genai"),
    "openai": package_version("openai"),
    "connection_checks": checks,
}
if checks["Google Drive"].startswith("PASS"):
    report_file = SETUP_DIR / ("connection_report_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + ".json")
    with report_file.open("x", encoding="utf-8") as f:
        json.dump(report, f, indent=2)
    assert json.loads(report_file.read_text(encoding="utf-8"))["google_cloud_project"] == PROJECT_ID
    print("Verified report saved:", report_file)
else:
    print("Report NOT SAVED: Drive not yet verified.")
print(json.dumps(report, indent=2))


## Execution handoff and honest verification

- **GitHub is a code bridge, not an execution/identity bridge:** ChatGPT edits repository code via its GitHub connector; Colab loads that reviewed GitHub code. Colab does not receive your ChatGPT identity or authorizations.
- **Google services are an integrated environment, not universally auto-authenticated:** Drive mount and Cloud ADC are separate interactive authorization steps. Verify both on each relevant Colab runtime.
- **ChatGPT ↔ Google Drive:** the connected Drive app can read or update authorized files independently of the Colab drive mount. This connection does not grant a Colab notebook access to ChatGPT.
- **Google Drive remains authoritative** for data, research results and non-sensitive reports; GitHub is for code and documentation only.
- **Gemini:** the Colab UI assistant requires its own manual UI test. Vertex AI SDK initialization does not verify API activation or permissions.
- **OpenAI GPT inside Colab:** adding an API secret is an *optional* separate route. Having ChatGPT Plus or an active ChatGPT/GitHub/Drive connector is not the same as having OpenAI Platform API access.
- **Paid services:** no model call, billing change or API enablement is included. Never give GPT unrestricted shell, Drive deletion rights or a cloud administrator role.

### Suggested control loop
**GPT plans/code-reviews → GitHub commit/PR → Colab opens and executes with your approval → Drive archives results → GPT analyzes via connected Drive.**

All results are notional until this notebook has actually run. This repository does not itself prove Colab and Google Cloud authorization.
